# 09 — Evaluación de prompts de PORTERÍAS por score (yellow + blue)

**Objetivo:** elegir, por **score** del decoder SAM 3, el mejor prompt de texto
para cada portería: `yellow_zone` y `blue_zone`. Misma metodología de score que el
notebook 02.

**Por qué:** el reto tiene **dos** porterías (una amarilla, una azul). En el análisis
original solo se cubrió la amarilla; la **azul faltaba**. Aquí se cierra esa brecha.

**Importante:** este análisis NO sube nada a Supervisely; solo evalúa prompts sobre
frames. Se usan tomas de **cámara superior** (campo completo → ambas porterías
visibles); en tomas Meta-Glasses portrait la azul suele quedar fuera de cuadro.

In [1]:
import os, sys
REPO=os.path.abspath('../..'); sys.path.insert(0, REPO); os.chdir(REPO)
import numpy as np, decord
decord.bridge.set_bridge('native')
from src.core.sam3_loader import load_sam3
from src.core.segmentation import segment_with_text
bundle = load_sam3()
print('SAM3 listo en', bundle.device)

Loading weights:   0%|          | 0/1797 [00:00<?, ?it/s]

SAM3 listo en cuda


## 1. Prompts candidatos + frames de prueba (cámara superior)

In [2]:
CANDS = {
    'yellow': ['yellow zone', 'yellow goal', 'yellow board', 'yellow plastic'],
    'blue':   ['blue zone', 'blue board', 'blue plastic', 'blue goal', 'dark blue rectangle', 'blue rectangle'],
}
VIDEOS = [
    'data/raw/18abril/Camara_superior/IMG_9933.MOV',
    'data/raw/18abril/Camara_superior/IMG_9938.MOV',
]
N = 8  # frames por video

frames = []
for v in VIDEOS:
    vr = decord.VideoReader(v)
    for i in np.linspace(int(len(vr)*0.1), int(len(vr)*0.9), N).astype(int):
        frames.append(vr[int(i)].asnumpy())
print(f'{len(frames)} frames de {len(VIDEOS)} videos')

16 frames de 2 videos


## 2. Evaluación por score

Por prompt: en cuántos frames detecta, score promedio (mejor detección por frame) y
área promedio. `score` = confianza del decoder SAM 3 (no es IoU).

In [3]:
rows = []
for goal, prompts in CANDS.items():
    for p in prompts:
        nf, scores, areas = 0, [], []
        for f in frames:
            dets = segment_with_text(f, p, bundle)
            if dets:
                nf += 1
                best = max(dets, key=lambda d: int(d.mask.sum()))
                scores.append(best.score); areas.append(int(best.mask.sum()))
        rows.append((goal, p, nf, len(frames),
                     float(np.mean(scores)) if scores else 0.0,
                     int(np.mean(areas)) if areas else 0))

print(f'{"goal":7s} {"prompt":22s} {"det/frames":>11s} {"avg_score":>10s} {"avg_area":>9s}')
print('-'*64)
for goal, p, nf, tot, s, a in rows:
    print(f'{goal:7s} {p:22s} {nf:>6d}/{tot:<4d} {s:>10.3f} {a:>9d}')

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

goal    prompt                  det/frames  avg_score  avg_area
----------------------------------------------------------------
yellow  yellow zone                16/16        0.851     33730
yellow  yellow goal                 0/16        0.000         0
yellow  yellow board               16/16        0.891     33799
yellow  yellow plastic             14/16        0.634     33947
blue    blue zone                   0/16        0.000         0
blue    blue board                  5/16        0.562     32368
blue    blue plastic                9/16        0.536     31708
blue    blue goal                   0/16        0.000         0
blue    dark blue rectangle        14/16        0.650     28488
blue    blue rectangle              4/16        0.524     28633


In [4]:
print('=== MEJOR por score (detección en >=50% frames) ===')
for goal in CANDS:
    cands = [r for r in rows if r[0]==goal and r[2] >= 0.5*len(frames)]
    if cands:
        best = max(cands, key=lambda r: r[4])
        print(f"  {goal}: '{best[1]}'  score={best[4]:.3f}  det={best[2]}/{best[3]}")
    else:
        print(f'  {goal}: ninguno detecta en >=50% frames')

=== MEJOR por score (detección en >=50% frames) ===
  yellow: 'yellow board'  score=0.891  det=16/16
  blue: 'dark blue rectangle'  score=0.650  det=14/16


## 3. Conclusión

Resultados (16 frames, 2 videos de cámara superior):

| Portería | Prompt | det/frames | avg_score |
|---|---|---:|---:|
| yellow | `yellow board` | 16/16 | **0.891** |
| yellow | `yellow zone` | 16/16 | 0.851 |
| yellow | `yellow plastic` | 14/16 | 0.634 |
| yellow | `yellow goal` | 0/16 | — |
| blue | **`dark blue rectangle`** | **14/16** | **0.650** |
| blue | `blue plastic` | 9/16 | 0.536 |
| blue | `blue board` | 5/16 | 0.562 |
| blue | `blue rectangle` | 4/16 | 0.524 |
| blue | `blue zone` / `blue goal` | 0/16 | — |

**Decisión:**
- `blue_zone` → **`dark blue rectangle`** (el más consistente: 14/16; `blue zone` no
  detecta nunca). Aplicado en `configs/01_yolo_sam3_config.json`.
- `yellow_zone` → se mantiene `yellow zone` (0.851) por continuidad con las
  anotaciones existentes; `yellow board` (0.891) es una mejora opcional.

Con esto ambas porterías entran al pipeline (segmentación, tracking, homografía de
fase_4).